# PIPELINE - Limpieza ENAHO

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

## Módulo 100: hogar

In [2]:
# carga del módulo 100
enaho_100 = pd.read_spss('../data/raw/Enaho01-2025-100.sav')

In [3]:
enaho_100.shape

(44599, 336)

In [4]:
# principales características del módulo 100
print('Módulo 100 (sin procesar) - Características generales de la vivienda y del hogar:')
print(f'Número de registros: {enaho_100.shape[0]}')
print(f'Número de variables: {enaho_100.shape[1]}')

Módulo 100 (sin procesar) - Características generales de la vivienda y del hogar:
Número de registros: 44599
Número de variables: 336


### Selección de variables de interés

In [5]:
variables_modulo_100 = [
    # --- IDENTIFICACIÓN Y CONTEXTO GEOGRÁFICO ---
    "AÑO",           # Año de la encuesta
    "MES",           # Mes de la encuesta
    "UBIGEO",        # Ubicación geográfica (Distrito)
    "CONGLOME",      # Conglomerado
    "VIVIENDA",      # Número de vivienda
    "HOGAR",         # Número de hogar
    "DOMINIO",       # Dominio Geográfico (Costa, Sierra, Selva, etc.)
    "ESTRATO",       # Estrato Geográfico / Tamaño de la población
    "FACTOR07",      # Factor de expansión anual

    # --- INFRAESTRUCTURA Y ENTORNO DE LA VIVIENDA ---
    "P101",          # Tipo de vivienda
    #"P102",          # Material predominante en las paredes exteriores
    #"P103",          # Material predominante en los pisos
    #"P103A",         # Material predominante en los techos
    #"P104",          # Cantidad de habitaciones en total en la vivienda
    #"P104A",         # Cantidad de habitaciones usadas exclusivamente para dormir

    # --- CONECTIVIDAD Y CANALES DE COMUNICACIÓN EXTERNA ---
    "P1142",         # El hogar tiene: Teléfono Celular
    "P1144",         # El hogar tiene: Conexión a Internet (Fijo/Móvil)

    # --- SERVICIOS BÁSICOS (ESTRESORES DOMÉSTICOS) ---
    #"P110",          # Procedencia principal del agua que utilizan en el hogar
    #"T110",          # (Recodificado) Procedencia del agua que utilizan en el hogar
    "P110C",         # ¿El hogar tiene acceso al servicio de agua todos los días de la semana?
    # "P110C1",      # (excluida a pedido) Horas de agua al día si tiene todos los días
    # "P110C3",      # (excluida a pedido) Horas de agua al día si no tiene todos los días
    "P1121",         # Tipo de alumbrado del hogar: Electricidad

    # --- BLOQUE COMPLETO DE NECESIDADES BÁSICAS INSATISFECHAS (NBI) ---
    "NBI1",          # Vivienda inadecuada (NBI 1)
    "NBI2",          # Vivienda con hacinamiento (NBI 2)
    "NBI3",          # Hogares con vivienda sin servicios higiénicos (NBI 3)
    "NBI4",          # Hogares con niños que no asisten a la escuela (NBI 4)
    "NBI5"           # Hogares con alta dependencia económica (NBI 5)
]

enaho_100 = enaho_100[variables_modulo_100]

print(f'Módulo 100 (filtrado) - Características generales de la vivienda y del hogar:')
print(f'Número de registros: {enaho_100.shape[0]}')
print(f'Número de variables: {enaho_100.shape[1]}')

Módulo 100 (filtrado) - Características generales de la vivienda y del hogar:
Número de registros: 44599
Número de variables: 19


### Diagnostico de columnas

In [6]:


# =====================================================================
# CELDA 2 (OPCIONAL / DIAGNÓSTICO): Auditoría de tipos de variable
# =====================================================================
# Este bloque NO modifica enaho_100, solo te ayuda a inspeccionar cuántas
# columnas son binarias, categóricas o numéricas. Las listas que genera
# (binary_cols, categorical_cols, numeric_cols, skipped_cols) NO se usan
# en las celdas siguientes, porque ahí la recodificación se hace a mano
# columna por columna. Puedes saltarte esta celda sin afectar el resto
# del notebook; la dejo solo como chequeo de auditoría.

total_columns_shape = enaho_100.shape[1]

binary_cols = []
categorical_cols = []
numeric_cols = []
skipped_cols = []

administrative_cols = ['AÑO', 'MES','UBIGEO', 'CONGLOME', 'VIVIENDA', 'HOGAR', 'FACTOR07']

for col in enaho_100.columns:
    if col in administrative_cols:
        skipped_cols.append(col)
        continue

    unique_vals = enaho_100[col].dropna().unique()
    num_unique = len(unique_vals)

    if num_unique == 0:
        skipped_cols.append(col)
    elif num_unique == 1:
        skipped_cols.append(col)
    elif num_unique == 2:
        binary_cols.append(col)
    elif enaho_100[col].dtype in ['object', 'category'] or num_unique < 15:
        categorical_cols.append(col)
    else:
        numeric_cols.append(col)

total_accounted_for = len(binary_cols) + len(categorical_cols) + len(numeric_cols) + len(skipped_cols)
all_accounted_columns = binary_cols + categorical_cols + numeric_cols + skipped_cols
missing_columns = [col for col in enaho_100.columns if col not in all_accounted_columns]

print(f"Total de columnas previstas: {total_columns_shape}")
print(f"Total de columnas encontradas:    {total_accounted_for}")
print("-" * 50)
print(f"Columnas binarias:      {len(binary_cols)}")
print(f"Columnas categóricas: {len(categorical_cols)}")
print(f"Columnas numéricas:     {len(numeric_cols)}")
print(f"Administrativas/Vacías: {len(skipped_cols)}")
print("-" * 50)

if len(missing_columns) > 0:
    print(f"⚠️ WARNING: There are {len(missing_columns)} columns completely missing from your lists!")
    print(f"The missing column names are: {missing_columns}")
else:
    print("La cantidad de columnas coincide. No hay columnas sin clasificar.")



Total de columnas previstas: 19
Total de columnas encontradas:    19
--------------------------------------------------
Columnas binarias:      9
Columnas categóricas: 3
Columnas numéricas:     0
Administrativas/Vacías: 7
--------------------------------------------------
La cantidad de columnas coincide. No hay columnas sin clasificar.


### Conversión de valores en columnas binarias

In [7]:

# =====================================================================
# CELDA 3: Recodificación binaria — servicios y NBI
# =====================================================================
# A partir de aquí trabajamos sobre una copia explícita, enaho_100_recoded,
# que irá acumulando TODAS las recodificaciones (esta celda y la siguiente).

enaho_100_recoded = enaho_100.copy()

# 1. Tecnología y servicios: 1.0 = tiene el servicio, 0.0 = no lo tiene ('Pase')
tech_utilities_maps = {
    'P1142': {'Telefono Celular': 1.0, 'Pase': 0.0},
    'P1144': {'Conexion a Internet': 1.0, 'Pase': 0.0},
    'P1121': {'Electricidad': 1.0, 'Pase': 0.0},
    'P110C': {'Si': 1.0, 'No': 0.0}
}

for col, mapping in tech_utilities_maps.items():
    if col in enaho_100_recoded.columns:
        enaho_100_recoded[col] = enaho_100_recoded[col].map(mapping).astype(float)

# 2. Necesidades Básicas Insatisfechas (NBI)
# 1.0 = presenta la carencia/vulnerabilidad, 0.0 = situación adecuada.
nbi_maps = {
    'NBI1': {'Vivienda inadecuada': 1.0, 'Vivienda adecuada': 0.0},
    'NBI2': {'Vivienda con hacinamiento': 1.0, 'Vivienda sin hacinamiento': 0.0},
    'NBI3': {'Hogares con vivienda sin servicios higienicos': 1.0, 'Hogares con vivienda con servicios higienicos': 0.0},
    'NBI4': {'Hogares con niños que no asisten a la escuela': 1.0, 'Hogares con niños que asisten a la escuela': 0.0},
    'NBI5': {'Hogares con alta dependencia economica': 1.0, 'Hogares sin alta dependencia economica': 0.0}
}

for col, mapping in nbi_maps.items():
    if col in enaho_100_recoded.columns:
        enaho_100_recoded[col] = enaho_100_recoded[col].map(mapping).astype(float)

print("✅ Recodificación binaria de servicios y NBI completa.")



✅ Recodificación binaria de servicios y NBI completa.


### Conversión de valores en columnas categóricas


In [8]:

# =====================================================================
# CELDA 4: Recodificación numérica discreta y categórica de vivienda
# =====================================================================
# IMPORTANTE: seguimos sobre enaho_100_recoded (resultado de la Celda 3),
# no sobre una copia nueva del original, para no perder lo ya recodificado.


enaho_100_recoded['MES'] = pd.to_numeric(enaho_100_recoded['MES'], errors='coerce').astype(float)
# 2. Variables categóricas de vivienda (1.0 = adecuado/noble, 0.0 = vulnerable/otro)

# P101: Tipo de vivienda
enaho_100_recoded["P101"] = (
    enaho_100_recoded["P101"]
    .isin(["Casa independiente", "Departamento en edificio"])
    .astype(float)
)

"""
# 1. Variables numéricas discretas
numeric_scale_cols = ["P104", "P104A"]
for col in numeric_scale_cols:
    enaho_100_recoded[col] = pd.to_numeric(enaho_100_recoded[col], errors="coerce").astype(float)



# P102: Material de paredes
enaho_100_recoded["P102"] = (
    enaho_100_recoded["P102"]
    .isin(["Ladrillo o bloque de cemento", "Piedra o sillar con cal o cemento"])
    .astype(float)
)

# P103: Material de pisos
enaho_100_recoded["P103"] = (
    enaho_100_recoded["P103"]
    .isin(
        [
            "Cemento",
            "Losetas, terrazos o similares",
            "Laminas asfalticas, vinilicos o similares",
            "Parquet o madera pulida",
        ]
    )
    .astype(float)
)

# P103A: Material de techos
enaho_100_recoded["P103A"] = (
    enaho_100_recoded["P103A"].isin(["Concreto armado", "Tejas"]).astype(float)
)

# P110: Infraestructura de agua
enaho_100_recoded["P110"] = (
    enaho_100_recoded["P110"]
    .isin(
        [
            "Red publica, dentro de la vivienda",
            "Red publica, fuera de la vivienda pero dentro del edificio",
        ]
    )
    .astype(float)
)

print("✅ Recodificación de variables categóricas y numéricas completada.")
print(f"Forma final del dataframe: {enaho_100_recoded.shape}")

"""

'\n# 1. Variables numéricas discretas\nnumeric_scale_cols = ["P104", "P104A"]\nfor col in numeric_scale_cols:\n    enaho_100_recoded[col] = pd.to_numeric(enaho_100_recoded[col], errors="coerce").astype(float)\n\n\n\n# P102: Material de paredes\nenaho_100_recoded["P102"] = (\n    enaho_100_recoded["P102"]\n    .isin(["Ladrillo o bloque de cemento", "Piedra o sillar con cal o cemento"])\n    .astype(float)\n)\n\n# P103: Material de pisos\nenaho_100_recoded["P103"] = (\n    enaho_100_recoded["P103"]\n    .isin(\n        [\n            "Cemento",\n            "Losetas, terrazos o similares",\n            "Laminas asfalticas, vinilicos o similares",\n            "Parquet o madera pulida",\n        ]\n    )\n    .astype(float)\n)\n\n# P103A: Material de techos\nenaho_100_recoded["P103A"] = (\n    enaho_100_recoded["P103A"].isin(["Concreto armado", "Tejas"]).astype(float)\n)\n\n# P110: Infraestructura de agua\nenaho_100_recoded["P110"] = (\n    enaho_100_recoded["P110"]\n    .isin(\n        [

### Renombramiento de variables

In [9]:
import pandas as pd

# Diccionario de mapeo con nombres cortos en minúsculas y español
diccionario_renombrar = {
    # --- IDENTIFICACIÓN Y CONTEXTO GEOGRÁFICO ---
    "AÑO": "anio",
    "MES": "mes",
    "UBIGEO": "ubigeo",
    "CONGLOME": "conglome",
    "VIVIENDA": "vivienda",
    "HOGAR": "hogar",
    "DOMINIO": "dominio",
    "ESTRATO": "estrato",
    "FACTOR07": "factor",
    # --- INFRAESTRUCTURA Y ENTORNO DE LA VIVIENDA ---
    "P101": "viv_tipo",
    "P102": "viv_paredes",
    "P103": "viv_pisos",
    "P103A": "viv_techos",
    "P104": "viv_hab_total",
    "P104A": "viv_hab_dormir",
    # --- CONECTIVIDAD Y CANALES DE COMUNICACIÓN EXTERNA ---
    "P1142": "tic_celular",
    "P1144": "tic_internet",
    # --- SERVICIOS BÁSICOS (ESTRESORES DOMÉSTICOS) ---
    "P110": "agua_procedencia",
    "T110": "agua_procedencia_recod",
    "P110C": "agua_todos_dias",
    "P110C1": "agua_horas_si_todos_dias",
    "P110C3": "agua_horas_no_todos_dias",
    "P1121": "serv_electricidad",
    # --- BLOQUE COMPLETO DE NECESIDADES BÁSICAS INSATISFECHAS (NBI) ---
    "NBI1": "nbi_vivienda",
    "NBI2": "nbi_hacinamiento",
    "NBI3": "nbi_sshh",
    "NBI4": "nbi_educacion",
    "NBI5": "nbi_dependencia",
}

# Aplicar el renombrado solo a las columnas que existan en tu DataFrame
enaho_100_final = enaho_100_recoded.rename(columns=diccionario_renombrar)

print("✅ Columnas renombradas exitosamente.")
print("Nuevas columnas en el dataset:\n", list(enaho_100_final.columns))

✅ Columnas renombradas exitosamente.
Nuevas columnas en el dataset:
 ['anio', 'mes', 'ubigeo', 'conglome', 'vivienda', 'hogar', 'dominio', 'estrato', 'factor', 'viv_tipo', 'tic_celular', 'tic_internet', 'agua_todos_dias', 'serv_electricidad', 'nbi_vivienda', 'nbi_hacinamiento', 'nbi_sshh', 'nbi_educacion', 'nbi_dependencia']


## Módulo 200: Miembros del hogar

In [10]:
enaho_200 = pd.read_spss('../data/raw/Enaho01-2025-200.sav')

### Selección de variables de interés

In [11]:
variables_modulo_200 = [
    # --- VARIABLES DE IDENTIFICACIÓN Y CONTEXTO ---
    "AÑO",         # Año de la encuesta (Control temporal)
    "MES",         # Mes de ejecución de la encuesta (Control estacional)
    "CONGLOME",    # Número de conglomerado (Llave de combinación con Módulo 100)
    "VIVIENDA",    # Número de selección de la vivienda (Llave de combinación con Módulo 100)
    "HOGAR",       # Número secuencial del hogar (Llave de combinación con Módulo 100)
    "CODPERSO",    # Número de orden de la persona (Identificador único intra-hogar)
    "UBIGEO",      # Ubicación geográfica (Establece distritos y regiones)
    "DOMINIO",     # Dominio Geográfico (Agrupación regional macros: Costa, Sierra, Selva)
    "ESTRATO",     # Estrato Geográfico / Tamaño poblacional (Densidad urbana vs rural)
 
    # --- CARACTERÍSTICAS INDIVIDUALES CRÍTICAS ---
    "P203",        # Relación de parentesco con el jefe(a) del hogar
    "P204",        # ¿Es miembro del hogar?
    "P207",        # Sexo
    "P208A",       # ¿Qué edad tiene en años cumplidos?
    "P209",        # ¿Cuál es su estado civil o conyugal?
 
    # --- CONDICIÓN DE ACTIVIDAD (LÍNEA BASE TRABAJO) ---
    #"P210",        # La semana pasada... ¿Estuvo trabajando o realizando tareas para obtener ingresos?
    "FACPOB07"     # Factor de expansión anual de población
]
 
enaho_200 = enaho_200[variables_modulo_200]
print(f"✅ Selección completa. Forma del dataframe: {enaho_200.shape}")


✅ Selección completa. Forma del dataframe: (115145, 15)


### Diagnóstico de columnas

In [12]:

total_columns_shape = enaho_200.shape[1]
 
binary_cols = []
categorical_cols = []
numeric_cols = []
skipped_cols = []
 
administrative_cols = ['ubigeo', 'factor', 'conglome', 'vivienda', 'hogar', 'codperso', 'estrato', 'dominio']
 
for col in enaho_200.columns:
    if col.lower() in administrative_cols:
        skipped_cols.append(col)
        continue
 
    unique_vals = enaho_200[col].dropna().unique()
    num_unique = len(unique_vals)
 
    if num_unique == 0:
        skipped_cols.append(col)
    elif num_unique == 1:
        skipped_cols.append(col)
    elif num_unique == 2:
        binary_cols.append(col)
    elif enaho_200[col].dtype in ['object', 'category'] or num_unique < 15:
        categorical_cols.append(col)
    else:
        numeric_cols.append(col)
 
total_accounted_for = len(binary_cols) + len(categorical_cols) + len(numeric_cols) + len(skipped_cols)
all_accounted_columns = binary_cols + categorical_cols + numeric_cols + skipped_cols
missing_columns = [col for col in enaho_200.columns if col not in all_accounted_columns]
 
print(f"Total Columns according to enaho_200.shape[1]: {total_columns_shape}")
print(f"Total Columns accounted for by loop:    {total_accounted_for}")
print("-" * 45)
print(f"🔗 Binary Columns:      {len(binary_cols)}")
print(f"📊 Categorical Columns: {len(categorical_cols)}")
print(f"📈 Numeric Columns:     {len(numeric_cols)}")
print(f"⚙️ Administrative/Empty: {len(skipped_cols)}")
print("-" * 45)
 
if len(missing_columns) > 0:
    print(f"⚠️ WARNING: There are {len(missing_columns)} columns completely missing from your lists!")
    print(f"The missing column names are: {missing_columns}")
else:
    print("✅ Success! The math balances perfectly. Every column is accounted for.")
 


Total Columns according to enaho_200.shape[1]: 15
Total Columns accounted for by loop:    15
---------------------------------------------
🔗 Binary Columns:      2
📊 Categorical Columns: 3
📈 Numeric Columns:     2
⚙️ Administrative/Empty: 8
---------------------------------------------
✅ Success! The math balances perfectly. Every column is accounted for.


### Conversión de valores en columnas numéricas

In [13]:

enaho_200_recoded = enaho_200.copy()
 
# MES: mes de ejecución de la encuesta, escala 1-12
enaho_200_recoded["MES"] = pd.to_numeric(enaho_200_recoded["MES"], errors="coerce").astype(float)
 
# P208A: edad en años cumplidos, escala continua
enaho_200_recoded["P208A"] = pd.to_numeric(enaho_200_recoded["P208A"], errors="coerce").astype(float)
 
print("✅ Conversión de variables numéricas continuas completada.")


✅ Conversión de variables numéricas continuas completada.


### Conversión de valores en columnas binarias

In [14]:
 
# P204: Miembro del hogar (Si = 1.0, No = 0.0)
enaho_200_recoded["P204"] = enaho_200_recoded["P204"].map({"Si": 1.0, "No": 0.0}).astype(float)
 
# P207: Sexo (Mujer = 1.0, Hombre = 0.0) -> orientado al proyecto de violencia de género
enaho_200_recoded["P207"] = enaho_200_recoded["P207"].map({"Mujer": 1.0, "Hombre": 0.0}).astype(float)
 
# P210: ¿Trabajó la semana pasada? (Si = 1.0, No = 0.0)
# enaho_200_recoded["P210"] = enaho_200_recoded["P210"].map({"Si": 1.0, "No": 0.0}).astype(float)
 
print("✅ Recodificación binaria de P204 y P207 completada.")
 


✅ Recodificación binaria de P204 y P207 completada.


### Conversión de valores en columnas categóricas

In [15]:
enaho_200_recoded["P203"] = enaho_200_recoded["P203"].isin(["Jefe/Jefa"]).astype(float)
 
# P209 (Estado civil): Casado(a) o Conviviente pasa a 1.0, el resto a 0.0 (luego "en_pareja")
enaho_200_recoded["P209"] = enaho_200_recoded["P209"].isin(["Casado(a)", "Conviviente"]).astype(float)
 
print("✅ Recodificación de P203 y P209 completada.")
print(f"Estructura actual (nombres originales intactos): {enaho_200_recoded.shape}")
 


✅ Recodificación de P203 y P209 completada.
Estructura actual (nombres originales intactos): (115145, 15)


### Renombrar columnas

In [16]:

diccionario_renombrar_200 = {
    # --- VARIABLES DE IDENTIFICACIÓN Y CONTEXTO ---
    "AÑO": "anio",
    "MES": "mes",
    "CONGLOME": "conglome",
    "VIVIENDA": "vivienda",
    "HOGAR": "hogar",
    "CODPERSO": "cod_persona",
    "UBIGEO": "ubigeo",
    "DOMINIO": "dominio",
    "ESTRATO": "estrato",
 
    # --- CARACTERÍSTICAS INDIVIDUALES MAPEADAS ---
    "P203": "es_jefe",               # 1.0 si es Jefe/Jefa de hogar
    "P204": "ind_miembro_hogar",     # 1.0 si es residente habitual
    "P207": "es_mujer",              # 1.0 si es mujer
    "P208A": "edad",                 # Escala continua en años
    "P209": "en_pareja",             # 1.0 si es casado(a) o conviviente
 
    # --- CONDICIÓN DE ACTIVIDAD Y EXPANSIÓN ---
    "P210": "trabajo_ingresos",      # 1.0 si trabajó la semana pasada
    "FACPOB07": "factor_poblacion"   # Factor de expansión poblacional (individuos)
}
 
enaho_200_final = enaho_200_recoded.rename(columns=diccionario_renombrar_200)
 
print("✅ Módulo 200: columnas renombradas exitosamente.")
print("Nuevas columnas en el dataset 200:\n", list(enaho_200_final.columns))


✅ Módulo 200: columnas renombradas exitosamente.
Nuevas columnas en el dataset 200:
 ['anio', 'mes', 'conglome', 'vivienda', 'hogar', 'cod_persona', 'ubigeo', 'dominio', 'estrato', 'es_jefe', 'ind_miembro_hogar', 'es_mujer', 'edad', 'en_pareja', 'factor_poblacion']


## Módulo 300: educación

In [17]:
enaho_300 = pd.read_spss('../data/raw/Enaho01A-2025-300.sav')

### Selección de variables de interés

In [18]:
variables_modulo_300 = [
    # --- LLAVES INDISPENSABLES DE FUSIÓN (Fusión con Módulos 100 y 200) ---
    "AÑO",          # Año de la encuesta (Control temporal)
    "MES",          # Mes de ejecución de la encuesta
    "CONGLOME",     # Número de conglomerado
    "VIVIENDA",     # Número de selección de la vivienda
    "HOGAR",        # Número secuencial del hogar
    "DOMINIO",      # Dominio Geográfico (Costa, Sierra, Selva, etc.)
    "ESTRATO",      # Estrato Geográfico / Tamaño de la población
    "CODPERSO",     # Número de orden de la persona (Match exacto por individuo)
    "UBIGEO",       # Ubicación geográfica (Distrito)
    
    # --- LOGRO EDUCATIVO Y CAPITAL HUMANO (Línea Base del Individuo) ---
    "P301A",        # Último año, grado de estudios o nivel educativo aprobado (Categoría clave)

    # --- ALFABETIZACIÓN FUNCIONAL Y LINGÜÍSTICA ---
    "P302",          # ¿Sabe leer y escribir? (Alfabetización funcional)
    "P300A",          # Lengua materna
     
    # --- TECNOLOGÍAS DE LA INFORMACIÓN Y ALFABETIZACIÓN DIGITAL ---
    "P316$1",       # Uso de Internet en los últimos 3 meses (Conectividad individual)
    "P316A1",       # Uso de teléfono celular propio en el mes anterior (Autonomía de comunicación)

    # --- FACTOR DE PONDERACIÓN INDIVIDUAL ---
    "FACTORA07"     # Factor de expansión anual de población del Módulo de Educación
]

enaho_300 = enaho_300[variables_modulo_300]

### Diagnóstico inicial de columnas

In [19]:

total_columns_shape = enaho_300.shape[1]
 
binary_cols = []
categorical_cols = []
numeric_cols = []
skipped_cols = []
 
administrative_cols = ['ubigeo', 'factor', 'conglome', 'vivienda', 'hogar', 'codperso', 'estrato', 'dominio']
 
for col in enaho_300.columns:
    if col.lower() in administrative_cols:
        skipped_cols.append(col)
        continue
 
    unique_vals = enaho_300[col].dropna().unique()
    num_unique = len(unique_vals)
 
    if num_unique == 0:
        skipped_cols.append(col)
    elif num_unique == 1:
        skipped_cols.append(col)
    elif num_unique == 2:
        binary_cols.append(col)
    elif enaho_300[col].dtype in ['object', 'category'] or num_unique < 15:
        categorical_cols.append(col)
    else:
        numeric_cols.append(col)
 
total_accounted_for = len(binary_cols) + len(categorical_cols) + len(numeric_cols) + len(skipped_cols)
all_accounted_columns = binary_cols + categorical_cols + numeric_cols + skipped_cols
missing_columns = [col for col in enaho_300.columns if col not in all_accounted_columns]
 
print(f"Total Columns according to enaho_300.shape[1]: {total_columns_shape}")
print(f"Total Columns accounted for by loop:    {total_accounted_for}")
print("-" * 45)
print(f"🔗 Binary Columns:      {len(binary_cols)}")
print(f"📊 Categorical Columns: {len(categorical_cols)}")
print(f"📈 Numeric Columns:     {len(numeric_cols)}")
print(f"⚙️ Administrative/Empty: {len(skipped_cols)}")
print("-" * 45)
 
if len(missing_columns) > 0:
    print(f"⚠️ WARNING: There are {len(missing_columns)} columns completely missing from your lists!")
    print(f"The missing column names are: {missing_columns}")
else:
    print("✅ Success! The math balances perfectly. Every column is accounted for.")
 


Total Columns according to enaho_300.shape[1]: 15
Total Columns accounted for by loop:    15
---------------------------------------------
🔗 Binary Columns:      3
📊 Categorical Columns: 3
📈 Numeric Columns:     1
⚙️ Administrative/Empty: 8
---------------------------------------------
✅ Success! The math balances perfectly. Every column is accounted for.


### Conversión de valores en columnas numéricas

In [20]:

enaho_300_recoded = enaho_300.copy()
 
# MES: mes de ejecución de la encuesta, escala 1-12
enaho_300_recoded["MES"] = pd.to_numeric(enaho_300_recoded["MES"], errors="coerce").astype(float)
 
print("✅ Conversión de MES a numérico continuo completada.")
 


✅ Conversión de MES a numérico continuo completada.


### Conversión de valores en columnas binarias

In [21]:
enaho_300_recoded['P302'] = enaho_300_recoded['P302'].fillna('Si')

enaho_300_recoded[['P302']].value_counts(dropna=False)

P302
Si      91558
No      12888
Name: count, dtype: int64

In [22]:

# P316$1: Habilidad / uso digital (Si = 1.0, No = 0.0)
enaho_300_recoded["P316$1"] = (
    enaho_300_recoded["P316$1"].map({"Si": 1.0, "No": 0.0}).astype(float)
)
 
# P316A1: Acceso a Internet vía celular propio (Telf.celular propio. = 1.0, Pase = 0.0)
enaho_300_recoded["P316A1"] = (
    enaho_300_recoded["P316A1"]
    .map({"Telf.celular propio.": 1.0, "Pase": 0.0})
    .astype(float)
)
 
enaho_300_recoded['P302'] = enaho_300_recoded['P302'].map({'Si': 1.0, 'No': 0.0}).astype(float)

print("✅ Recodificación binaria de P316$1 y P316A1 completada.")
 


✅ Recodificación binaria de P316$1 y P316A1 completada.


### Conversión de valores en columnas categóricas

In [23]:

educacion_raw = enaho_300_recoded["P301A"]
 
# Nivel superior: tecnológico, universitario o posgrado (completo o incompleto)
lista_superior = [
    "Superior no univ. completa",
    "Superior univ. completa",
    "Superior univ. incompleta",
    "Superior no univ. Incompleta",
    "Maestria / Doctorado"
]
enaho_300_recoded["P301A"] = educacion_raw.isin(lista_superior).astype(float)
 
# Nivel básico: el máximo nivel alcanzado quedó dentro del marco escolar básico
lista_basica = [
    "Educacion inicial",
    "Primaria incompleta",
    "Primaria completa",
    "Secundaria incompleta",
    "Secundaria completa",
    "Basica especial"
]
enaho_300_recoded["P301A_basica"] = educacion_raw.isin(lista_basica).astype(float)


lista_nativas = [
    "Quechua",
    "Aimara",
    "Ashaninka",
    "Awajun/Aguarun",
    "Shipibo – Konibo",
    "Shawi/Chayahuita",
    "Matsigenka/Machiguenga",
    "Achuar",
    "Otra lengua nativa",
]
enaho_300_recoded["P300A_nativa"] = enaho_300_recoded["P300A"].isin(lista_nativas).astype(float)



print("✅ Variables de nivel educativo (superior / básica) calculadas correctamente.")
print(f"Estructura actual del DataFrame 300: {enaho_300_recoded.shape}")
 


✅ Variables de nivel educativo (superior / básica) calculadas correctamente.
Estructura actual del DataFrame 300: (104446, 17)


### Renombrar variables

In [24]:
diccionario_renombrar_300 = {
    # --- LLAVES INDISPENSABLES DE FUSIÓN ---
    "AÑO": "anio",
    "MES": "mes",
    "CONGLOME": "conglome",
    "VIVIENDA": "vivienda",
    "HOGAR": "hogar",
    "DOMINIO": "dominio",
    "ESTRATO": "estrato",
    "CODPERSO": "cod_persona",
    "UBIGEO": "ubigeo",
 
    # --- LOGRO EDUCATIVO Y CAPITAL HUMANO ---
    "P301A": "edu_superior",         # 1.0 si alcanzó nivel superior o posgrado
    "P301A_basica": "edu_basica",    # 1.0 si su máximo nivel fue el marco escolar básico

    # --- ALFABETIZACIÓN FUNCIONAL Y LINGÜÍSTICA ---
    "P302": "alfabetizacion_funcional",  # 1.0 si sabe leer y escribir (alfabetización funcional)
    "P300A": "lengua_materna",           # Lengua materna (categoría original, se puede recodificar a nativa vs no nativa)
    "P300A_nativa": "lengua_materna_nativa",  # 1.0 si la lengua materna es una lengua nativa peruana
 
    # --- TECNOLOGÍAS DE LA INFORMACIÓN Y ALFABETIZACIÓN DIGITAL ---
    "P316$1": "tic_uso_internet",    # 1.0 si usó internet en los últimos 3 meses
    "P316A1": "tic_celular_propio",  # 1.0 si usó celular propio el mes anterior
 
    # --- FACTOR DE PONDERACIÓN INDIVIDUAL ---
    "FACTORA07": "factor_educacion"  # Factor de expansión de este módulo
}
 
enaho_300_final = enaho_300_recoded.rename(columns=diccionario_renombrar_300)
enaho_300_final.drop(columns=["lengua_materna"], inplace=True)  # Eliminamos la columna original de lengua materna, ya que ahora tenemos la versión nativa vs no nativa
 
print("✅ Módulo 300: columnas renombradas exitosamente.")
print("Nuevas columnas en el dataset 300:\n", list(enaho_300_final.columns))


✅ Módulo 300: columnas renombradas exitosamente.
Nuevas columnas en el dataset 300:
 ['anio', 'mes', 'conglome', 'vivienda', 'hogar', 'dominio', 'estrato', 'cod_persona', 'ubigeo', 'edu_superior', 'alfabetizacion_funcional', 'tic_uso_internet', 'tic_celular_propio', 'factor_educacion', 'edu_basica', 'lengua_materna_nativa']


## Módulo 400: salud

In [25]:
enaho_400 = pd.read_spss('../data/raw/Enaho01A-2025-400.sav')

In [26]:
print(list(enaho_400.columns))

['AÑO', 'MES', 'CONGLOME', 'VIVIENDA', 'HOGAR', 'CODPERSO', 'UBIGEO', 'DOMINIO', 'ESTRATO', 'CODINFOR', 'P400N', 'P400I', 'P400A1', 'P400A2', 'P400A3', 'P401C', 'P401D1', 'P401D2', 'P401D3', 'P401D4', 'P401D5', 'P401D6', 'P401D7', 'P401D8', 'P401D9', 'P401F', 'P401G', 'P401G1', 'P401G2', 'P401H1', 'P401H2', 'P401H3', 'P401H4', 'P401H5', 'P401H6', 'P401', 'P4021', 'P4022', 'P4023', 'P4024', 'P4025', 'P4026', 'P4031', 'P4032', 'P4033', 'P4034', 'P4035', 'P4036', 'P4037', 'P4038', 'P4039', 'P40310', 'P40311', 'P40313', 'P40314', 'P4041', 'P4042', 'P4043', 'P4044', 'P4045', 'P4046', 'P4047', 'P4091', 'P4092', 'P4093', 'P4094', 'P4095', 'P4096', 'P4097', 'P4098', 'P4099', 'P40910', 'P40911', 'P413B1', 'P413B1A', 'P413B2', 'P413B2A', 'P413D1', 'P413D1A', 'P413D2', 'P413D2A', 'P414N$01', 'P414N$02', 'P414N$03', 'P414N$04', 'P414N$05', 'P414N$06', 'P414N$07', 'P414N$08', 'P414N$09', 'P414N$10', 'P414N$11', 'P414N$12', 'P414N$13', 'P414N$14', 'P414N$15', 'P414N$16', 'P414$01', 'P414$02', 'P414$

### Selección de variables de interés

In [27]:
variables_modulo_400 = [
    # --- LLAVES INDISPENSABLES ---
    "AÑO", "MES", "CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO", "UBIGEO",
    
    # --- CONTROLES SOCIODEMOGRÁFICOS PRESENTES ---
    "DOMINIO", "ESTRATO", "P203", "P207", "P208A", "P209",
    
    # --- ENFERMEDAD CRÓNICA ---
    "P401",  # ¿Sufre de alguna enfermedad o malestar crónico?
    
    # --- DISCAPACIDAD / LIMITACIONES PERMANENTES (Serie P401H) ---
    "P401H1",  # Moverse o caminar / usar brazos o piernas
    "P401H2",  # Ver, aun usando anteojos
    "P401H3",  # Hablar o comunicarse
    "P401H4",  # Oír, aún usando audífonos
    "P401H5",  # Entender o aprender (concentrarse y recordar)
    "P401H6",  # Relacionarse con los demás (emociones/conductas)
    
    # --- SEGUROS DE SALUD ---
    "P4191",  # ¿Tiene ESSALUD?
    "P4192",  # ¿Tiene seguro privado de salud?
    "P4193",  # ¿Tiene seguro de EPS?
    "P4194",  # ¿Tiene Seguro de FFAA?
    "P4195",  # ¿Tiene Seguro Integral de Salud?
    "P4196",  # ¿Tiene seguro universitario?
    "P4197",  # ¿Tiene seguro escolar privado?
    "P4198",  # ¿Tiene algún otro tipo de seguro?
    
    # --- FACTOR DE EXPANSION DISPONIBLE EN ESTE ARCHIVO ---
    "FACTOR07" 
]



enaho_400 = enaho_400[variables_modulo_400]

### Diagnóstico inicial de columnas

In [28]:

total_columns_shape = enaho_400.shape[1]
 
binary_cols = []
categorical_cols = []
numeric_cols = []
skipped_cols = []
 
administrative_cols = ['ubigeo', 'factor', 'conglome', 'vivienda', 'hogar', 'codperso', 'estrato', 'dominio']
 
for col in enaho_400.columns:
    if col.lower() in administrative_cols:
        skipped_cols.append(col)
        continue
 
    unique_vals = enaho_400[col].dropna().unique()
    num_unique = len(unique_vals)
 
    if num_unique == 0:
        skipped_cols.append(col)
    elif num_unique == 1:
        skipped_cols.append(col)
    elif num_unique == 2:
        binary_cols.append(col)
    elif enaho_400[col].dtype in ['object', 'category'] or num_unique < 15:
        categorical_cols.append(col)
    else:
        numeric_cols.append(col)
 
total_accounted_for = len(binary_cols) + len(categorical_cols) + len(numeric_cols) + len(skipped_cols)
all_accounted_columns = binary_cols + categorical_cols + numeric_cols + skipped_cols
missing_columns = [col for col in enaho_400.columns if col not in all_accounted_columns]
 
print(f"Total Columns according to enaho_400.shape[1]: {total_columns_shape}")
print(f"Total Columns accounted for by loop:    {total_accounted_for}")
print("-" * 45)
print(f"🔗 Binary Columns:      {len(binary_cols)}")
print(f"📊 Categorical Columns: {len(categorical_cols)}")
print(f"📈 Numeric Columns:     {len(numeric_cols)}")
print(f"⚙️ Administrative/Empty: {len(skipped_cols)}")
print("-" * 45)
 
if len(missing_columns) > 0:
    print(f"⚠️ WARNING: There are {len(missing_columns)} columns completely missing from your lists!")
    print(f"The missing column names are: {missing_columns}")
else:
    print("✅ Success! The math balances perfectly. Every column is accounted for.")
 


Total Columns according to enaho_400.shape[1]: 29
Total Columns accounted for by loop:    29
---------------------------------------------
🔗 Binary Columns:      16
📊 Categorical Columns: 3
📈 Numeric Columns:     2
⚙️ Administrative/Empty: 8
---------------------------------------------
✅ Success! The math balances perfectly. Every column is accounted for.


### Conversión de valores en columnas binarias

In [29]:
enaho_400_recoded = enaho_400.copy()
 
# P207: Sexo (Mujer = 1.0, Hombre = 0.0)
enaho_400_recoded["P207"] = enaho_400_recoded["P207"].map({"Mujer": 1.0, "Hombre": 0.0}).astype(float)
 
# P401 y desagregaciones P401H (Si = 1.0, No = 0.0)
columnas_sintomas = ["P401", "P401H1", "P401H2", "P401H3", "P401H4", "P401H5", "P401H6"]
for col in columnas_sintomas:
    if col in enaho_400_recoded.columns:
        enaho_400_recoded[col] = enaho_400_recoded[col].map({"Si": 1.0, "No": 0.0}).astype(float)
 
# Cobertura de seguros (P419): 1.0 = tiene el seguro específico, 0.0 = No
mapeos_seguros = {
    "P4191": {"EsSalud": 1.0, "No": 0.0},
    "P4192": {"Seguro Privado de Salud": 1.0, "No": 0.0},
    "P4193": {"Entidad Prestadora de Salud": 1.0, "No": 0.0},
    "P4194": {"Seguro FF.AA./PNP": 1.0, "No": 0.0},
    "P4195": {"Seguro Integral de Salud (SIS)": 1.0, "No": 0.0},
    "P4196": {"Seguro Universitario": 1.0, "No": 0.0},
    "P4197": {"Seguro Escolar Privado": 1.0, "No": 0.0},
    "P4198": {"Otro": 1.0, "No": 0.0}
}
 
for col, mapa in mapeos_seguros.items():
    if col in enaho_400_recoded.columns:
        enaho_400_recoded[col] = enaho_400_recoded[col].map(mapa).astype(float)
 
print("✅ Recodificación binaria de sexo, síntomas/discapacidad y seguros completada.")
 


✅ Recodificación binaria de sexo, síntomas/discapacidad y seguros completada.


### Conversión de valores en columnas categóricas

In [30]:
 
# MES: mes de ejecución de la encuesta, escala 1-12
enaho_400_recoded["MES"] = pd.to_numeric(enaho_400_recoded["MES"], errors="coerce").astype(float)
 
# P203 (Parentesco): Jefe/Jefa pasa a 1.0, el resto a 0.0
enaho_400_recoded["P203"] = enaho_400_recoded["P203"].isin(["Jefe/Jefa"]).astype(float)
 
# P209 (Estado civil): Casado(a) o Conviviente pasa a 1.0, el resto a 0.0
enaho_400_recoded["P209"] = enaho_400_recoded["P209"].isin(["Casado(a)", "Conviviente"]).astype(float)
 
print("✅ Recodificación de MES, P203 y P209 completada.")
print(f"Estructura actual del DataFrame 400: {enaho_400_recoded.shape}")
 


✅ Recodificación de MES, P203 y P209 completada.
Estructura actual del DataFrame 400: (107802, 29)


### Renombrar columnas

In [31]:
# 1. Define la lista de columnas binarias que quieres evaluar
columnas_seguro = ['P4191', 'P4192', 'P4193', 'P4194', 'P4195', 'P4196', 'P4197', 'P4198']

# 2. Crea la nueva columna condicional
enaho_400_recoded['P419'] = enaho_400_recoded[columnas_seguro].any(axis=1).astype(int)
enaho_400_recoded.drop(columns=columnas_seguro, inplace=True)  # Elimina las columnas individuales de seguros si ya no las necesitas

columnas_limitacion = ['P401H1', 'P401H2', 'P401H3', 'P401H4', 'P401H5', 'P401H6']

enaho_400_recoded['P401H'] = enaho_400_recoded[columnas_limitacion].any(axis=1).astype(int)
enaho_400_recoded.drop(columns=columnas_limitacion, inplace=True)  # Elimina las columnas individuales de limitaciones si ya no las necesitas

In [32]:
 
diccionario_renombrar_400 = {
    # --- LLAVES INDISPENSABLES ---
    "AÑO": "anio",
    "MES": "mes",
    "CONGLOME": "conglome",
    "VIVIENDA": "vivienda",
    "HOGAR": "hogar",
    "CODPERSO": "cod_persona",
    "UBIGEO": "ubigeo",
 
    # --- CONTROLES SOCIODEMOGRÁFICOS ---
    "DOMINIO": "dominio",
    "ESTRATO": "estrato",
    "P203": "es_jefe",               # 1.0 si es Jefe/Jefa de hogar
    "P207": "es_mujer",              # 1.0 si es mujer
    "P208A": "edad",                 # Escala continua en años
    "P209": "en_pareja",             # 1.0 si es casado(a) o conviviente
 
    # --- ENFERMEDAD CRÓNICA ---
    "P401": "salud_enfermedad_cronica",
 
    # --- DISCAPACIDAD / LIMITACIONES PERMANENTES (Serie P401H) ---
    "P401H": "tiene_limitacion",
 
    # --- SEGUROS DE SALUD ---
    "P419": "tiene_seguro",
    # --- FACTOR DE EXPANSIÓN ---
    "FACTOR07": "factor_salud"
}
 
enaho_400_final = enaho_400_recoded.rename(columns=diccionario_renombrar_400)
 
print("✅ Módulo 400: columnas renombradas exitosamente.")
print(f"Dimensiones finales del dataset 400: {enaho_400_final.shape}")
print("Columnas actuales:\n", list(enaho_400_final.columns))


✅ Módulo 400: columnas renombradas exitosamente.
Dimensiones finales del dataset 400: (107802, 17)
Columnas actuales:
 ['anio', 'mes', 'conglome', 'vivienda', 'hogar', 'cod_persona', 'ubigeo', 'dominio', 'estrato', 'es_jefe', 'es_mujer', 'edad', 'en_pareja', 'salud_enfermedad_cronica', 'factor_salud', 'tiene_seguro', 'tiene_limitacion']


## Agregación de columnas

### filtrado de solo mujeres

In [33]:
llaves_individuo = ["conglome", "vivienda", "hogar"]
llaves_persona   = ["conglome", "vivienda", "hogar", "cod_persona"]

df_personas = pd.merge(enaho_200_final, enaho_300_final, on=llaves_persona, how="inner")

df_personas = pd.merge(df_personas, enaho_400_final, on=llaves_persona, how="inner")

In [34]:
df_personas[['es_mujer_x', 'es_mujer_y']].value_counts(dropna=False)

es_mujer_x  es_mujer_y
1.0         1.0           54063
0.0         0.0           50383
Name: count, dtype: int64

In [35]:
df_personas.drop(columns=["anio_x", "mes_x", "ubigeo_x", "dominio_x", "estrato_x", "es_jefe_x", "es_mujer_x", "edad_x", "en_pareja_x", "anio_y", "mes_y", "ubigeo_y", "dominio_y", "estrato_y"], inplace=True)
df_personas.rename(columns={"es_jefe_y": "es_jefe", 
                            "es_mujer_y": "es_mujer", 
                            "edad_y": "edad", 
                            "en_pareja_y": "en_pareja"}, inplace=True)

In [36]:
df_personas.shape

(104446, 26)

In [37]:
df_personas.columns

Index(['conglome', 'vivienda', 'hogar', 'cod_persona', 'ind_miembro_hogar',
       'factor_poblacion', 'edu_superior', 'alfabetizacion_funcional',
       'tic_uso_internet', 'tic_celular_propio', 'factor_educacion',
       'edu_basica', 'lengua_materna_nativa', 'anio', 'mes', 'ubigeo',
       'dominio', 'estrato', 'es_jefe', 'es_mujer', 'edad', 'en_pareja',
       'salud_enfermedad_cronica', 'factor_salud', 'tiene_seguro',
       'tiene_limitacion'],
      dtype='str')

In [38]:
mujeres_adultas = df_personas[(df_personas['es_mujer'] == 1.0) & (df_personas['edad'] >= 18)]

In [39]:
mujeres_adultas.shape

(40318, 26)

In [40]:
mujeres_adultas.to_csv('../data/processed/mujeres_adultas.csv', index=False)

### Función agregadora

In [171]:
def calcular_media_distrital_ponderada(df, columna_valor, columna_factor, columna_ubigeo='ubigeo'):
    """
    Calcula la media ponderada de una variable continua, agrupada por distrito.
    """
    df_limpio = df.dropna(subset=[columna_valor, columna_factor, columna_ubigeo])
    
    numerador = (df_limpio[columna_valor] * df_limpio[columna_factor]).groupby(df_limpio[columna_ubigeo]).sum()
    denominador = df_limpio.groupby(columna_ubigeo)[columna_factor].sum()
    
    return (numerador / denominador.replace(0, np.nan)).rename('media')

In [172]:
import pandas as pd
import numpy as np

def calcular_proporcion_distrital_ponderada(df, columna_binaria, columna_factor, columna_ubigeo='ubigeo'):
    """
    Calcula la proporción poblacional ponderada agrupada por distrito (ubigeo).

    Parámetros:
    df (DataFrame): Dataset filtrado (ej. Mujeres > 18).
    columna_binaria (str): Columna con valores 0 y 1.
    columna_factor (str): Factor de expansión del módulo respectivo.
    columna_ubigeo (str): Nombre de la columna de código distrital.

    Retorna:
    Series: Proporción ponderada indexada por cada ubigeo.
    """
    columnas_requeridas = [columna_binaria, columna_factor, columna_ubigeo]

    # 1. Eliminar filas con nulos en las variables críticas
    df_limpio = df.dropna(subset=columnas_requeridas)

    # 2. Numerador y denominador ponderados, agrupados por distrito en un solo paso
    numerador = (df_limpio[columna_binaria] * df_limpio[columna_factor]).groupby(df_limpio[columna_ubigeo]).sum()
    denominador = df_limpio.groupby(columna_ubigeo)[columna_factor].sum()

    # 3. Proporción final, evitando división por cero
    proporcion = (numerador / denominador.replace(0, np.nan)).rename('proporcion')

    return proporcion

In [173]:
# 1. Mapeamos cada variable a su factor de expansión correspondiente
variables_factor_binarias = {
    # CARACTERÍSTICAS DE LOS MIEMBROS DEL HOGAR (Módulo 200)
    'es_jefe': 'factor_poblacion',
    #'ind_miembro_hogar': 'factor_poblacion',
    'es_mujer': 'factor_poblacion',
    #'edad': 'factor_poblacion',
    'en_pareja': 'factor_poblacion',
    #'trabajo_ingresos': 'factor_poblacion',


    # EDUCACIÓN (Módulo 300)
    'edu_superior': 'factor_educacion',
    'edu_basica': 'factor_educacion',
    'alfabetizacion_funcional': 'factor_educacion',
    'lengua_materna_nativa': 'factor_educacion',
    'tic_uso_internet': 'factor_educacion',
    'tic_celular_propio': 'factor_educacion',

    # SALUD (Módulo 400)
    'salud_enfermedad_cronica': 'factor_salud',

    'tiene_limitacion': 'factor_salud',
    'tiene_seguro': 'factor_salud',
    #'lim_discapacidad_motriz': 'factor_salud',
    #'lim_discapacidad_visual': 'factor_salud',
    #'lim_discapacidad_habla': 'factor_salud',
    #'lim_discapacidad_auditiva': 'factor_salud',
    #'lim_discapacidad_comprension': 'factor_salud',
    #'lim_discapacidad_relacion': 'factor_salud',
    #'seg_essalud': 'factor_salud',
    #'seg_privado': 'factor_salud',
    #'seg_eps': 'factor_salud',
    #'seg_ffaa_pnp': 'factor_salud',
    #'seg_sis': 'factor_salud',
    #'seg_universitario': 'factor_salud',
    #'seg_escolar_privado': 'factor_salud',
    #'seg_otro': 'factor_salud'
    
}

variables_factor_continuas = {
    'edad': 'factor_poblacion',
}



In [174]:
# 2. Calculamos todas las proporciones en un solo loop y las unimos con concat
df_distritos_mujeres = pd.concat(
    {
        f'prop_{col}': calcular_proporcion_distrital_ponderada(
            df=mujeres_adultas,
            columna_binaria=col,
            columna_factor=factor
        )
        for col, factor in variables_factor_binarias.items()
    },
    axis=1
)

# 3. Calculamos las medias ponderadas para las variables continuas y las unimos
df_distritos_mujeres_continuas = pd.concat(
    {
        f'media_{col}': calcular_media_distrital_ponderada(
            df=mujeres_adultas,
            columna_valor=col,
            columna_factor=factor
        )
        for col, factor in variables_factor_continuas.items()
    },
    axis=1
)

# 4. Unimos ambos resultados en un solo DataFrame por distrito
df_distritos_mujeres = df_distritos_mujeres.join(df_distritos_mujeres_continuas, how='outer')
df_distritos_mujeres.index.name = 'ubigeo'

print(df_distritos_mujeres.head())

        prop_es_jefe  prop_es_mujer  prop_en_pareja  prop_edu_superior  \
ubigeo                                                                   
010101      0.334482            1.0        0.456272           0.426559   
010104      0.125000            1.0        0.625000           0.000000   
010106      0.285714            1.0        0.714286           0.000000   
010109      0.222794            1.0        0.602521           0.120571   
010110      0.393529            1.0        0.457313           0.369073   

        prop_edu_basica  prop_alfabetizacion_funcional  \
ubigeo                                                   
010101         0.522272                       0.937497   
010104         1.000000                       1.000000   
010106         1.000000                       1.000000   
010109         0.821234                       0.745002   
010110         0.464479                       0.828042   

        prop_lengua_materna_nativa  prop_tic_uso_internet  \
ubigeo       

In [175]:
df_distritos_mujeres.reset_index(inplace=True)

In [176]:
df_distritos_mujeres.to_csv('../data/processed/proporciones_distritales_mujeres_adultas.csv', index=True)

## enaho - hogares

In [177]:
variables_binarias_hogar = {'viv_tipo':'factor', 
                  #'viv_paredes':'factor', 
                  #'viv_pisos':'factor', 
                  #'viv_techos':'factor', 
                  'tic_celular':'factor',
                   'tic_internet':'factor', 
                   #'agua_procedencia':'factor',  
                   'agua_todos_dias':'factor', 
                   'serv_electricidad':'factor', 
                   'nbi_vivienda':'factor', 
                   'nbi_hacinamiento':'factor', 
                   'nbi_sshh':'factor', 
                   'nbi_educacion':'factor', 
                   'nbi_dependencia':'factor'}

#variables_continuas_hogar = {'viv_hab_total':'factor',
#                            'viv_hab_dormir':'factor'}

In [178]:
enaho_100_final.dtypes

anio                      str
mes                   float64
ubigeo                    str
conglome                  str
vivienda                  str
hogar                     str
dominio              category
estrato              category
factor                float64
viv_tipo              float64
tic_celular           float64
tic_internet          float64
agua_todos_dias       float64
serv_electricidad     float64
nbi_vivienda          float64
nbi_hacinamiento      float64
nbi_sshh              float64
nbi_educacion         float64
nbi_dependencia       float64
dtype: object

In [179]:
df_distritos_hogares = pd.concat(
    {
        f'prop_{col}': calcular_proporcion_distrital_ponderada(
            df=enaho_100_final,
            columna_binaria=col,
            columna_factor=factor
        )
        for col, factor in variables_binarias_hogar.items()
    },
    axis=1
)
"""
# 3. Calculamos las medias ponderadas para las variables continuas y las unimos
df_distritos_hogares_continuas = pd.concat(
    {
        f'media_{col}': calcular_media_distrital_ponderada(
            df=enaho_100_final,
            columna_valor=col,
            columna_factor=factor
        )
        for col, factor in variables_continuas_hogar.items()
    },
    axis=1
)
"""
# 4. Unimos ambos resultados en un solo DataFrame por distrito
#df_distritos_hogares = df_distritos_hogares.join(df_distritos_hogares_continuas, how='outer')
df_distritos_hogares.index.name = 'ubigeo'

df_distritos_hogares.reset_index(inplace=True)
print(df_distritos_hogares.head())

   ubigeo  prop_viv_tipo  prop_tic_celular  prop_tic_internet  \
0  010101       0.677290          0.942543           0.927752   
1  010104       0.666667          1.000000           1.000000   
2  010106       1.000000          0.875000           0.750000   
3  010109       0.737556          0.865025           0.865025   
4  010110       0.659159          0.972946           0.972946   

   prop_agua_todos_dias  prop_serv_electricidad  prop_nbi_vivienda  \
0              0.731953                0.962448           0.029381   
1              1.000000                1.000000           0.000000   
2              1.000000                0.750000           0.000000   
3              1.000000                0.932512           0.383417   
4              1.000000                0.837675           0.162325   

   prop_nbi_hacinamiento  prop_nbi_sshh  prop_nbi_educacion  \
0               0.014239       0.035045                 0.0   
1               0.000000       0.000000                 0.0   

In [180]:
df_distritos_hogares.describe()

,prop_viv_tipo,prop_tic_celular,prop_tic_internet,prop_agua_todos_dias,prop_serv_electricidad,prop_nbi_vivienda,prop_nbi_hacinamiento,prop_nbi_sshh,prop_nbi_educacion,prop_nbi_dependencia
count,1271.000000,1271.000000,1271.000000,1233.000000,1271.000000,1271.000000,1271.000000,1271.000000,1271.000000,1271.000000
mean,0.742745,0.911555,0.868399,0.870839,0.914738,0.081162,0.044270,0.102379,0.003775,0.005642
std,0.157002,0.121480,0.160670,0.251623,0.180603,0.151911,0.079749,0.164642,0.020292,0.026458
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.652768,0.875000,0.833258,0.869279,0.895907,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.750000,0.954799,0.918571,1.000000,1.000000,0.000000,0.000000,0.021391,0.000000,0.000000
75%,0.857143,1.000000,1.000000,1.000000,1.000000,0.109882,0.063622,0.133578,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.750000,1.000000,0.285713,0.375000


### unir datasets

In [181]:
# unir dataset distrital de mujeres con características del hoghar
df_distritos_mujeres_hogares = df_distritos_hogares.merge(df_distritos_mujeres, on='ubigeo', how='inner')

In [182]:
df_distritos_mujeres_hogares.head()

,ubigeo,prop_viv_tipo,prop_tic_celular,prop_tic_internet,prop_agua_todos_dias,prop_serv_electricidad,prop_nbi_vivienda,prop_nbi_hacinamiento,prop_nbi_sshh,prop_nbi_educacion,...,prop_edu_superior,prop_edu_basica,prop_alfabetizacion_funcional,prop_lengua_materna_nativa,prop_tic_uso_internet,prop_tic_celular_propio,prop_salud_enfermedad_cronica,prop_tiene_limitacion,prop_tiene_seguro,media_edad
0,010101,0.677290,0.942543,0.927752,0.731953,0.962448,0.029381,0.014239,0.035045,0.0,...,0.426559,0.522272,0.937497,0.0,0.838800,0.941529,0.679256,0.112473,0.980843,47.066860
1,010104,0.666667,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,1.000000,1.000000,0.0,0.914349,0.736773,0.500000,0.000000,1.000000,47.250000
2,010106,1.000000,0.875000,0.750000,1.000000,0.750000,0.000000,0.000000,0.000000,0.0,...,0.000000,1.000000,1.000000,0.0,NaN,1.000000,0.714286,0.142857,1.000000,59.285714
3,010109,0.737556,0.865025,0.865025,1.000000,0.932512,0.383417,0.000000,0.076683,0.0,...,0.120571,0.821234,0.745002,0.0,0.813716,0.784441,0.579753,0.000000,1.000000,46.969642
4,010110,0.659159,0.972946,0.972946,1.000000,0.837675,0.162325,0.000000,0.000000,0.0,...,0.369073,0.464479,0.828042,0.0,0.866948,0.981205,0.830911,0.027976,1.000000,52.438633


In [183]:
df_distritos_mujeres_hogares.to_csv('../data/processed/proporciones_distritales_mujeres_hogares.csv', index=True)

### anterior - eliminar

In [154]:
df_distritos_mujeres.describe()

,prop_es_jefe,prop_ind_miembro_hogar,prop_es_mujer,prop_edad,prop_en_pareja,prop_trabajo_ingresos,prop_edu_superior,prop_edu_basica,prop_alfabetizacion_funcional,prop_lengua_materna_nativa,...,prop_lim_discapacidad_comprension,prop_lim_discapacidad_relacion,prop_seg_essalud,prop_seg_privado,prop_seg_eps,prop_seg_ffaa_pnp,prop_seg_sis,prop_seg_universitario,prop_seg_escolar_privado,prop_seg_otro
count,1271.000000,1271.000000,1271.0,1271.000000,1271.000000,0.0,1271.000000,1271.000000,1271.000000,1271.000000,...,1271.000000,1271.000000,1271.000000,1271.000000,1271.000000,1271.000000,1271.000000,1271.000000,1271.0,1271.000000
mean,0.299761,0.996704,1.0,48.926946,0.576326,NaN,0.224900,0.673086,0.856309,0.333904,...,0.018320,0.007234,0.139121,0.003694,0.002560,0.003818,0.807931,0.001770,0.0,0.002335
std,0.151753,0.014936,0.0,7.088967,0.165225,NaN,0.195033,0.185816,0.149277,0.374775,...,0.044202,0.024908,0.161992,0.021312,0.016654,0.017524,0.193309,0.011721,0.0,0.021002
min,0.000000,0.857143,1.0,29.333333,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
25%,0.200000,1.000000,1.0,43.844863,0.460753,NaN,0.000000,0.542827,0.794415,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.698541,0.000000,0.0,0.000000
50%,0.297998,1.000000,1.0,47.804199,0.567203,NaN,0.206608,0.674291,0.901623,0.141886,...,0.000000,0.000000,0.097767,0.000000,0.000000,0.000000,0.857143,0.000000,0.0,0.000000
75%,0.378048,1.000000,1.0,53.436508,0.667783,NaN,0.359734,0.818440,0.965009,0.674314,...,0.012154,0.000000,0.222611,0.000000,0.000000,0.000000,1.000000,0.000000,0.0,0.000000
max,1.000000,1.000000,1.0,74.750000,1.000000,NaN,0.879029,1.000000,1.000000,1.000000,...,0.500000,0.250000,1.000000,0.296361,0.285714,0.333333,1.000000,0.200000,0.0,0.600000


In [ ]:
columnas_miembro  = ['es_jefe', 'ind_miembro_hogar', 'es_mujer', 'edad', 'en_pareja', 'trabajo_ingresos', 'factor_poblacion']

In [ ]:
columnas_educacion = ['edu_superior', 'edu_basica', 'alfabetizacion_funcional', 'lengua_materna_nativa', 'tic_uso_internet', 'tic_celular_propio', 'factor_educacion']

In [127]:
enaho_400_final.columns

Index(['anio', 'mes', 'conglome', 'vivienda', 'hogar', 'cod_persona', 'ubigeo',
       'dominio', 'estrato', 'es_jefe', 'es_mujer', 'edad', 'en_pareja',
       'salud_enfermedad_cronica', 'lim_discapacidad_motriz',
       'lim_discapacidad_visual', 'lim_discapacidad_habla',
       'lim_discapacidad_auditiva', 'lim_discapacidad_comprension',
       'lim_discapacidad_relacion', 'seg_essalud', 'seg_privado', 'seg_eps',
       'seg_ffaa_pnp', 'seg_sis', 'seg_universitario', 'seg_escolar_privado',
       'seg_otro', 'factor_salud'],
      dtype='str')

In [93]:
llaves_individuo = ["anio", "mes", "conglome", "vivienda", "hogar", "ubigeo", "dominio", "estrato"]
llaves_persona   = ["anio", "mes", "conglome", "vivienda", "hogar", "ubigeo", "dominio", "estrato", "cod_persona"]

df_personas = pd.merge(enaho_200_final, enaho_300_final, on=llaves_persona, how="inner")

df_personas = pd.merge(df_personas, enaho_400_final, on=llaves_persona, how="inner")

In [109]:
enaho_200_final.shape

(115145, 16)

In [110]:
df_personas.shape

(42228, 43)

In [100]:
enaho_200_final.columns

Index(['anio', 'mes', 'conglome', 'vivienda', 'hogar', 'cod_persona', 'ubigeo',
       'dominio', 'estrato', 'es_jefe', 'ind_miembro_hogar', 'es_mujer',
       'edad', 'en_pareja', 'trabajo_ingresos', 'factor_poblacion'],
      dtype='str')

In [95]:
df_personas[['factor_poblacion', 'factor_educacion', 'factor_salud']].describe()

,factor_poblacion,factor_educacion,factor_salud
count,42228.000000,42228.000000,42228.000000
mean,247.957943,243.368515,247.957943
std,187.253714,219.021486,187.253714
min,1.882153,0.192009,1.882153
25%,136.325531,99.086227,136.325531
50%,209.929977,185.789200,209.929977
75%,321.794647,320.651428,321.794647
max,1236.256958,3748.197266,1236.256958


In [ ]:
import numpy as np
import pandas as pd

# =====================================================================
# 1. FUSIÓN DE LOS MÓDULOS A NIVEL DE INDIVIDUO (MERGE)
# =====================================================================

# Llaves de combinación por hogar e individuo
llaves_individuo = ["anio", "conglome", "vivienda", "hogar", "ubigeo"]
llaves_persona   = ["anio", "conglome", "vivienda", "hogar", "ubigeo", "cod_persona"]

# Módulo 200 + Módulo 300 (nivel persona)
df_personas = pd.merge(enaho_200_final, enaho_300_final, on=llaves_persona, how="inner")

# + Módulo 400 (salud) — excluir columnas ya presentes en 200 para evitar duplicados _x/_y
cols_salud_clean = [
    c
    for c in enaho_400_final.columns
    if c not in ["dominio", "estrato", "factor_salud"] or c in llaves_persona
]
df_personas = pd.merge(
    df_personas, enaho_400_final[cols_salud_clean], on=llaves_persona, how="inner"
)

# + Módulo 100 (vivienda/hogar) — se une por llave de hogar (sin cod_persona)
enaho_individual = pd.merge(
    df_personas, enaho_100_final, on=llaves_individuo, how="inner"
)

print(f"✅ Fusión completada. Registros individuales consolidados: {enaho_individual.shape}")


# =====================================================================
# 2. COLAPSO DE COLUMNAS DUPLICADAS (_x / _y) TRAS EL MERGE
# =====================================================================
def _coalesce_column_variants(df, base_name):
    variants = [c for c in df.columns if c == base_name or c.startswith(f"{base_name}_")]
    if not variants:
        return df
    if base_name in df.columns and len(variants) == 1:
        return df
    df = df.copy()
    df[base_name] = df[variants].bfill(axis=1).iloc[:, 0]
    drop_cols = [c for c in variants if c != base_name]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

for col_base in ["es_jefe", "es_mujer", "edad", "en_pareja"]:
    enaho_individual = _coalesce_column_variants(enaho_individual, col_base)

if "es_jefe" not in enaho_individual.columns and "P203" in enaho_individual.columns:
    enaho_individual = enaho_individual.rename(columns={"P203": "es_jefe"})


# =====================================================================
# 3. DEFINICIÓN DE VARIABLES PARA LA AGREGACIÓN POR DISTRITO
# =====================================================================
variables_a_agregar = [
    # Módulo 100 — Vivienda y servicios del hogar
    "viv_tipo",
    "viv_paredes",
    "viv_pisos",
    "viv_techos",
    "viv_hab_total",
    "viv_hab_dormir",
    "tic_celular",
    "tic_internet",
    "agua_todos_dias",
    "serv_electricidad",
    "nbi_vivienda",
    "nbi_hacinamiento",
    "nbi_sshh",
    "nbi_educacion",
    "nbi_dependencia",
    # Módulo 200 — Características del individuo
    "es_jefe",
    "es_mujer",
    "edad",
    "en_pareja",
    "trabajo_ingresos",
    # Módulo 300 — Educación y acceso digital
    "edu_superior",
    "edu_basica",
    "tic_uso_internet",
    "tic_celular_propio",
    # Módulo 400 — Salud y cobertura
    "salud_enfermedad_cronica",
    "lim_discapacidad_motriz",
    "lim_discapacidad_visual",
    "lim_discapacidad_habla",
    "lim_discapacidad_auditiva",
    "lim_discapacidad_comprension",
    "lim_discapacidad_relacion",
    "seg_essalud",
    "seg_privado",
    "seg_sis",
]

enaho_individual = enaho_individual[(enaho_individual.edad >= 18) & (enaho_individual.es_mujer == 1.0)]

MergeError: Passing 'suffixes' which cause duplicate columns {'mes_x', 'mes_y'} is not allowed.

In [ ]:

# =====================================================================
# 4. AGREGACIÓN PONDERADA POR DISTRITO (ubigeo)
# =====================================================================
def _agregar_distrito_ponderado(group):
    """Calcula la media ponderada por factor de expansión para cada variable."""
    pesos = group["factor_poblacion"]
    resultados = {}

    for col in variables_a_agregar:
        mascara_valida = group[col].notna()
        if not mascara_valida.any():
            resultados[col] = np.nan
            continue
        valores_validos = group.loc[mascara_valida, col]
        pesos_validos   = pesos.loc[mascara_valida]
        suma_pesos      = pesos_validos.sum()
        resultados[col] = (
            (valores_validos * pesos_validos).sum() / suma_pesos
            if suma_pesos > 0
            else np.nan
        )
    return pd.Series(resultados)


# Filtrar solo las variables que existen en el dataframe
variables_disponibles = [c for c in variables_a_agregar if c in enaho_individual.columns]

# Detectar columna de ponderación (cualquier columna que contenga 'factor')
candidatos_peso = [c for c in enaho_individual.columns if "factor" in c.lower()]
columna_peso = candidatos_peso[0] if candidatos_peso else None

def _collapse_group(group):
    if not variables_disponibles:
        return pd.Series(dtype=float)
    if columna_peso and columna_peso in group.columns:
        sub = group[variables_disponibles + [columna_peso]].rename(
            columns={columna_peso: "factor_poblacion"}
        )
    else:
        sub = group[variables_disponibles].copy()
        sub["factor_poblacion"] = 1.0
    return _agregar_distrito_ponderado(sub)

enaho_por_distrito = (
    enaho_individual.groupby("ubigeo")
    .apply(_collapse_group, include_groups=False)
    .reset_index()
)

print("Peso usado:", columna_peso if columna_peso else "ninguno (pesos iguales)")
print(f"Variables agregadas: {variables_disponibles}")
print(f"✅ Dataset final por distrito: {enaho_por_distrito.shape}")


In [34]:
# =====================================================================
# 5. FILTRO DE CONFIABILIDAD MUESTRAL POR DISTRITO
# =====================================================================

UMBRAL_HOGARES  = 20   # mínimo de hogares distintos encuestados por distrito
UMBRAL_PERSONAS = 30   # mínimo de personas distintas encuestadas por distrito
UMBRAL_MUJERES  = 10   # mínimo de mujeres (para variables de género)

# Calcular tamaños muestrales reales (sin ponderar) por distrito
conteo_hogares = (
    enaho_individual
    .drop_duplicates(subset=["ubigeo", "conglome", "vivienda", "hogar"])
    .groupby("ubigeo")
    .size()
    .rename("n_hogares")
)

conteo_personas = (
    enaho_individual
    .groupby("ubigeo")
    .size()
    .rename("n_personas")
)

conteo_mujeres = (
    enaho_individual[enaho_individual["es_mujer"] == 1.0]
    .groupby("ubigeo")
    .size()
    .rename("n_mujeres")
)

# Unir conteos al dataset distrital
enaho_por_distrito = (
    enaho_por_distrito
    .join(conteo_hogares,  on="ubigeo")
    .join(conteo_personas, on="ubigeo")
    .join(conteo_mujeres,  on="ubigeo")
)

# Rellenar NaN en n_mujeres (distritos sin mujeres en muestra)
enaho_por_distrito["n_mujeres"] = enaho_por_distrito["n_mujeres"].fillna(0).astype(int)

# Aplicar filtro
mascara_confiable = (
    (enaho_por_distrito["n_hogares"]  >= UMBRAL_HOGARES)  &
    (enaho_por_distrito["n_personas"] >= UMBRAL_PERSONAS) &
    (enaho_por_distrito["n_mujeres"]  >= UMBRAL_MUJERES)
)

enaho_distritos_confiables = enaho_por_distrito[mascara_confiable].copy()

n_total    = len(enaho_por_distrito)
n_retenido = len(enaho_distritos_confiables)

print(f"Distritos totales:    {n_total}")
print(f"Distritos retenidos:  {n_retenido}  ({n_retenido/n_total:.1%})")
print(f"Distritos descartados: {n_total - n_retenido}")
print(f"\nRango n_hogares:  {enaho_por_distrito['n_hogares'].min()}–{enaho_por_distrito['n_hogares'].max()}")
print(f"Rango n_personas: {enaho_por_distrito['n_personas'].min()}–{enaho_por_distrito['n_personas'].max()}")

Distritos totales:    1271
Distritos retenidos:  411  (32.3%)
Distritos descartados: 860

Rango n_hogares:  3–433
Rango n_personas: 6–1378


In [38]:
enaho_distritos_confiables.describe()

,viv_tipo,viv_paredes,viv_pisos,viv_techos,viv_hab_total,viv_hab_dormir,tic_celular,tic_internet,agua_todos_dias,serv_electricidad,...,lim_discapacidad_habla,lim_discapacidad_auditiva,lim_discapacidad_comprension,lim_discapacidad_relacion,seg_essalud,seg_privado,seg_sis,n_hogares,n_personas,n_mujeres
count,411.000000,411.000000,411.000000,411.000000,411.000000,411.000000,411.000000,411.000000,409.000000,411.000000,...,411.000000,411.000000,411.000000,411.000000,411.000000,411.000000,411.000000,411.000000,411.000000,411.000000
mean,0.946949,0.516675,0.636119,0.384557,3.431299,2.136621,0.965659,0.939179,0.829805,0.941514,...,0.008048,0.014693,0.016900,0.009188,0.213016,0.010071,0.702557,61.141119,193.600973,100.523114
std,0.067125,0.291509,0.289943,0.292724,0.594122,0.495728,0.051748,0.070141,0.251238,0.119413,...,0.009377,0.017919,0.020343,0.012088,0.155897,0.034383,0.189239,62.119197,203.155453,106.979580
min,0.644081,0.000000,0.000000,0.000000,1.488158,0.655836,0.569603,0.501485,0.000000,0.104529,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.051591,20.000000,43.000000,21.000000
25%,0.915057,0.279753,0.414614,0.123134,3.061900,1.815404,0.959481,0.925263,0.774938,0.947061,...,0.000000,0.000000,0.003526,0.000000,0.085366,0.000000,0.590232,26.000000,78.000000,41.000000
50%,0.973923,0.554433,0.697744,0.371901,3.445200,2.179919,0.983484,0.961864,0.942122,0.990299,...,0.005680,0.010134,0.011914,0.006390,0.197912,0.000000,0.725543,38.000000,116.000000,61.000000
75%,1.000000,0.744166,0.883949,0.611685,3.813410,2.486414,0.994281,0.981738,1.000000,1.000000,...,0.012747,0.020752,0.023891,0.013987,0.304925,0.006725,0.847267,67.000000,215.500000,112.000000
max,1.000000,1.000000,1.000000,1.000000,5.231575,3.456948,1.000000,1.000000,1.000000,1.000000,...,0.049600,0.147401,0.228898,0.109383,0.700708,0.314061,1.000000,433.000000,1378.000000,723.000000


In [40]:
enaho_distritos_confiables = enaho_distritos_confiables.drop(columns=["n_hogares", "n_personas", "n_mujeres"])
print(f"✅ Dataset final de distritos confiables: {enaho_distritos_confiables.shape}")

✅ Dataset final de distritos confiables: (411, 35)


In [ ]:
enaho_distritos_confiables.to_csv('../data/processed/enaho_por_distrito.csv', index=False)